# **08. Model Evaluation**

#### **Introduction**

This notebook evaluates a predictive model developed using the **Bank Telemarketing dataset** containing 45,211 customer records. The model's performance is evaluated using **Gain, Lift, and Decile Analysis** to determine how effectively it identifies customers who are more likely to subscribe to a term deposit. The analysis helps in ranking customers based on their predicted response and identifying the most promising customer segments for targeted marketing.


In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sidetable
import sklearn
import feature_engine
import scipy
from scipy import stats
import time
from pathlib import Path
import pickle
import joblib

In [2]:
# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from IPython.display import display, Markdown

def display_md(string):
    display(Markdown(string))
    

In [6]:
# Define path for dataframe file
parent_path = Path.cwd().parent
data_file = parent_path.joinpath("data", "interim")
files = []
for file in data_file.rglob("*"):
    files.append(file)
    print(file.name)
    print(files.index(file), " ", file, end="\n\n")


.gitkeep
0   e:\Bank-Telemarketing\data\interim\.gitkeep

xgb_prediction.csv
1   e:\Bank-Telemarketing\data\interim\xgb_prediction.csv



In [ ]:
display_md("**Loading Dataframe of  Actual & Predicted Values...**")
try:
    df = pd.read_csv(files[1], sep=',')
    display(df.head())
except Exception as e:
    print(f"Data Loading Error : {e}")
    

**Loading Dataframe of  Actual & Predicted Values...**

,actual_xgb,probability_xgb
0,0,0.009562
1,0,0.888495
2,0,0.082937
3,0,0.953316
4,0,0.512221


## **1. Decile Analysis**

In [26]:
# First, sort customers by their predicted probability from highest to lowest

df_result = df.sort_values(
    by='probability_xgb',
    ascending=False
).reset_index(drop=True)

# Create 10 equal-sized groups
df_result['decile'] = pd.qcut(
    df_result['probability_xgb'],
    q=10,
    labels=False,
    duplicates='drop'
)

# Make highest probability decile = 1
df_result['decile'] = 10 - df_result['decile']

# Now calculate decile-level performance
decile_table = df_result.groupby('decile').agg(
    customers = ('actual_xgb', 'count'),
    responders = ('actual_xgb', 'sum'),
    response_rate = ('actual_xgb', 'mean')
).reset_index()

decile_table['response_rate'] = decile_table['response_rate'].mul(100)

decile_df = decile_table.style.format({"customers":"{:,.0f}", "response_rate":"{:.1f}%"}).hide(axis='index')
display(decile_df)


decile,customers,responders,response_rate
1,"1,492",907,60.8%
2,"1,492",526,35.3%
3,"1,492",193,12.9%
4,"1,492",73,4.9%
5,"1,491",27,1.8%
6,"1,492",11,0.7%
7,"1,492",4,0.3%
8,"1,492",1,0.1%
9,"1,492",2,0.1%
10,"1,492",1,0.1%
